#### This notebook uses the madllian architecture to ingest the data by pyspark and then make a transformation in sliver and then insert into rdbms to create demnesion model in gold layer 

In [1]:
!pip --version

pip 23.3 from /opt/conda/lib/python3.11/site-packages/pip (python 3.11)


In [2]:
!python3 -V

Python 3.11.6


In [3]:
!pip list

Package                       Version
----------------------------- ------------
alembic                       1.12.0
altair                        5.1.2
anyio                         4.0.0
argon2-cffi                   23.1.0
argon2-cffi-bindings          21.2.0
arrow                         1.3.0
asttokens                     2.4.0
async-generator               1.10
async-lru                     2.0.4
attrs                         23.1.0
Babel                         2.13.0
backcall                      0.2.0
backports.functools-lru-cache 1.6.5
beautifulsoup4                4.12.2
bleach                        6.1.0
blinker                       1.6.3
bokeh                         3.3.0
boltons                       23.0.0
Bottleneck                    1.3.7
Brotli                        1.1.0
cached-property               1.5.2
certifi                       2023.7.22
certipy                       0.1.3
cffi                          1.16.0
charset-normalizer            3.3.0
click   

In [4]:
!pip show pandas

Name: pandas
Version: 2.0.3
Summary: Powerful data structures for data analysis, time series, and statistics
Home-page: 
Author: 
Author-email: The Pandas Development Team <pandas-dev@python.org>
License: BSD 3-Clause License

Copyright (c) 2008-2011, AQR Capital Management, LLC, Lambda Foundry, Inc. and PyData Development Team
All rights reserved.

Copyright (c) 2011-2023, Open source contributors.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contributors may be used to endorse or promote

In [5]:
!pip show SQLAlchemy                    

Name: SQLAlchemy
Version: 2.0.22
Summary: Database Abstraction Library
Home-page: https://www.sqlalchemy.org
Author: Mike Bayer
Author-email: mike_mp@zzzcomputing.com
License: MIT
Location: /opt/conda/lib/python3.11/site-packages
Requires: greenlet, typing-extensions
Required-by: alembic, jupyterhub


In [6]:
!echo $JAVA_HOME

In [7]:
!which java 

/usr/bin/java


In [8]:
!java --version

openjdk 17.0.14 2025-01-21
OpenJDK Runtime Environment (build 17.0.14+7-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.14+7-Ubuntu-122.04.1, mixed mode, sharing)


In [9]:
!echo $USER

In [10]:
from pyspark.sql import SparkSession

In [11]:
spark = SparkSession.builder\
.appName('SparkHazem')\
.master('local[*]')\
.config('spark.driver.name','HazemSpark')\
.config('spark.driver.host' , '172.18.0.5')\
.config('spark.sql.catalogImplementation' , 'hive')\
.config('spark.jars' , '/usr/local/spark-3.5.0-bin-hadoop3/jars/postgresql-42.7.5.jar')\
.config('spark.jars' , '/usr/local/spark-3.5.0-bin-hadoop3/jars/mysql-connector-java-9.3.0.jar')\
.getOrCreate()

In [12]:
spark

In [13]:
sc = spark.sparkContext

In [14]:
sc

<SparkContext master=local[*] appName=SparkHazem>

In [15]:
sc.setLogLevel("ERROR")

In [16]:
!ipython3 --version

8.16.1


In [17]:
# !conda install IPython

In [18]:
from IPython.display import display,HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

In [19]:
from pyspark.sql.functions import *

In [20]:
from pyspark.sql.types import *

In [21]:
import pyspark.sql.functions as fn 

In [22]:
import pandas as pd 

### Bronz Layer : Ingest The data from csv by file parcing 

**ingest by pyspark the data from csv files**

`Note Analysis the data first before make a ingestion`

In [23]:
# read the customer information from file parcing (csv) | pull extraction
cust_info = spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_crm/cust_info.csv')\
.load()

In [24]:
# print schema to detected the data will uploaded with right infer schema 
cust_info.printSchema()

root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: date (nullable = true)



In [25]:
cust_info.dtypes

[('cst_id', 'int'),
 ('cst_key', 'string'),
 ('cst_firstname', 'string'),
 ('cst_lastname', 'string'),
 ('cst_marital_status', 'string'),
 ('cst_gndr', 'string'),
 ('cst_create_date', 'date')]

In [26]:
cust_info.show(2 , truncate=False)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|cst_key   |cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
|11000 |AW00011000| Jon         |Yang        |M                 |M       |2025-10-06     |
|11001 |AW00011001|Eugene       |Huang       |S                 |M       |2025-10-06     |
+------+----------+-------------+------------+------------------+--------+---------------+
only showing top 2 rows



In [27]:
# read the product information from file parcing (csv) | pull extraction
prod_info =  spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_crm/prd_info.csv')\
.load()

In [28]:
prod_info.show(2 ,truncate=False)

+------+----------------+-------------------------+--------+--------+------------+----------+
|prd_id|prd_key         |prd_nm                   |prd_cost|prd_line|prd_start_dt|prd_end_dt|
+------+----------------+-------------------------+--------+--------+------------+----------+
|210   |CO-RF-FR-R92B-58|HL Road Frame - Black- 58|NULL    |R       |2003-07-01  |NULL      |
|211   |CO-RF-FR-R92R-58|HL Road Frame - Red- 58  |NULL    |R       |2003-07-01  |NULL      |
+------+----------------+-------------------------+--------+--------+------------+----------+
only showing top 2 rows



In [29]:
# read the Sales information from file parcing (csv) | pull extraction
sales_info =  spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_crm/sales_details.csv')\
.load()

In [30]:
sales_info.printSchema()

root
 |-- sls_ord_num: string (nullable = true)
 |-- sls_prd_key: string (nullable = true)
 |-- sls_cust_id: integer (nullable = true)
 |-- sls_order_dt: integer (nullable = true)
 |-- sls_ship_dt: integer (nullable = true)
 |-- sls_due_dt: integer (nullable = true)
 |-- sls_sales: integer (nullable = true)
 |-- sls_quantity: integer (nullable = true)
 |-- sls_price: integer (nullable = true)



In [31]:
sales_info.show(2, truncate=False)

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|SO43697    |BK-R93R-62 |21768      |20101229    |20110105   |20110110  |3578     |1           |3578     |
|SO43698    |BK-M82S-44 |28389      |20101229    |20110105   |20110110  |3400     |1           |3400     |
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
only showing top 2 rows



In [32]:
# read the customer info birthdate added another info for customer from file parcing (csv) | pull extraction
cust_bdate =  spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_erp/CUST_AZ12.csv')\
.load()

In [33]:
cust_bdate.show(2, truncate=False)

+-------------+----------+----+
|CID          |BDATE     |GEN |
+-------------+----------+----+
|NASAW00011000|1971-10-06|Male|
|NASAW00011001|1976-05-10|Male|
+-------------+----------+----+
only showing top 2 rows



In [34]:
# read the customer info country added another info for customer from file parcing (csv) | pull extraction
cust_country =  spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_erp/LOC_A101.csv')\
.load()

In [35]:
cust_country.show(2, truncate=False)

+-----------+---------+
|CID        |CNTRY    |
+-----------+---------+
|AW-00011000|Australia|
|AW-00011001|Australia|
+-----------+---------+
only showing top 2 rows



In [36]:
prd_maintenace =  spark.read.format('csv')\
.option('inferSchema' , 'true') \
.option('header' , 'true') \
.option('samplingRatio' , 0.001)\
.option('path' , '/home/jovyan/work/source_erp/PX_CAT_G1V2.csv')\
.load()

In [37]:
prd_maintenace.show(2 , truncate=False)

+-----+-----------+-----------+-----------+
|ID   |CAT        |SUBCAT     |MAINTENANCE|
+-----+-----------+-----------+-----------+
|AC_BR|Accessories|Bike Racks |Yes        |
|AC_BS|Accessories|Bike Stands|No         |
+-----+-----------+-----------+-----------+
only showing top 2 rows



### ✅ Data Integration the first step in sliver layer 

![data_integration](data_integration/data_integration.png)
)

### Sliver Layer : Make a standralization and data cleansing and check about data quality

`check the data qulity first before make anythoing`

### ✅ Check the Validation of Data Completeness

- Ensure all expected rows are present in the dataset.
- Verify that no critical columns contain `NULL` or missing values.
- Compare record counts between source and target tables (if applicable).

In [38]:
cust_count = cust_info.select('*')\
.count()

In [39]:
cust_count

18494

In [40]:
sales_info.select('*')\
.count()

60398

In [41]:
prod_info.select('*')\
.count()

397

### Analysis for the data in customer table 

**exploration customer_info table**

**Have relation in customer dim in (cust_info & cust_bdate & cust_country)**

In [42]:
cust_info.dtypes

[('cst_id', 'int'),
 ('cst_key', 'string'),
 ('cst_firstname', 'string'),
 ('cst_lastname', 'string'),
 ('cst_marital_status', 'string'),
 ('cst_gndr', 'string'),
 ('cst_create_date', 'date')]

In [43]:
cust_bdate.dtypes

[('CID', 'string'), ('BDATE', 'date'), ('GEN', 'string')]

In [44]:
cust_country.dtypes

[('CID', 'string'), ('CNTRY', 'string')]

In [45]:
cust_info.show(2 , truncate=False)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|cst_key   |cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
|11000 |AW00011000| Jon         |Yang        |M                 |M       |2025-10-06     |
|11001 |AW00011001|Eugene       |Huang       |S                 |M       |2025-10-06     |
+------+----------+-------------+------------+------------------+--------+---------------+
only showing top 2 rows



In [46]:
cust_bdate.show(2, truncate=False)

+-------------+----------+----+
|CID          |BDATE     |GEN |
+-------------+----------+----+
|NASAW00011000|1971-10-06|Male|
|NASAW00011001|1976-05-10|Male|
+-------------+----------+----+
only showing top 2 rows



In [47]:
cust_info.show(truncate=False)

+------+----------+-------------+------------+------------------+--------+---------------+
|cst_id|cst_key   |cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|
+------+----------+-------------+------------+------------------+--------+---------------+
|11000 |AW00011000| Jon         |Yang        |M                 |M       |2025-10-06     |
|11001 |AW00011001|Eugene       |Huang       |S                 |M       |2025-10-06     |
|11002 |AW00011002|Ruben        | Torres     |M                 |M       |2025-10-06     |
|11003 |AW00011003|Christy      |  Zhu       |S                 |F       |2025-10-06     |
|11004 |AW00011004| Elizabeth   |Johnson     |S                 |F       |2025-10-06     |
|11005 |AW00011005|Julio        |Ruiz        |S                 |M       |2025-10-06     |
|11006 |AW00011006|Janet        |Alvarez     |S                 |F       |2025-10-06     |
|11007 |AW00011007|Marco        |Mehta       |M                 |M       |2025-10-06     |

In [48]:
cust_info_meta_data = cust_info.withColumn('dwh_cst_date' , current_timestamp())

In [49]:
cust_info_meta_data.show(3)

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:47:...|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2025-05-03 16:47:...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 16:47:...|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
only showing top 3 rows



In [50]:
cust_id_null = cust_info.select(col('cst_id'))\
.where(col('cst_id').isNull()).count()

In [51]:
cust_id_null

4

In [52]:
cust_id_replica = cust_info.select('*')\
.groupBy(col('cst_id'))\
.agg(fn.count('*').alias('cust_count'))\
.where((col('cust_count') > 1))


In [53]:
cust_id_replica.show()

+------+----------+
|cst_id|cust_count|
+------+----------+
| 29483|         2|
|  NULL|         4|
| 29473|         2|
| 29449|         2|
| 29466|         3|
| 29433|         2|
+------+----------+



In [54]:
cust_info_check_cst_id  = cust_info_meta_data.select('*') \
.where(col('cst_id')== 29466).show(3)

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 29466|AW00029466|         NULL|        NULL|              NULL|    NULL|     2026-01-25|2025-05-03 16:47:...|
| 29466|AW00029466|        Lance|     Jimenez|                 M|    NULL|     2026-01-26|2025-05-03 16:47:...|
| 29466|AW00029466|        Lance|     Jimenez|                 M|       M|     2026-01-27|2025-05-03 16:47:...|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+



In [55]:
cust_info_view = cust_info_meta_data.createOrReplaceTempView('customer_information')

In [56]:
cust_info_vw_test = spark.sql('SELECT * FROM customer_information ').show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:47:...|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2025-05-03 16:47:...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 16:47:...|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2025-05-03 16:

In [57]:
cust_info_edit_qulity_cst_id=spark.sql ('''
    SELECT ROW_NUMBER() over(partition by cst_id order by cst_create_date DESC) as rn,
    crm.cst_id , crm.cst_key , crm.cst_firstname , crm.cst_lastname , crm.cst_marital_status , crm.cst_gndr,
    crm.cst_create_date , crm.dwh_cst_date
    FROM customer_information as crm 
''')

In [58]:
cust_info_edit_qulity_cst_id.show()

+---+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| rn|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+---+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|  1|  NULL|     SF566|         NULL|        NULL|              NULL|    NULL|           NULL|2025-05-03 16:47:...|
|  2|  NULL|      PO25|         NULL|        NULL|              NULL|    NULL|           NULL|2025-05-03 16:47:...|
|  3|  NULL|  13451235|         NULL|        NULL|              NULL|    NULL|           NULL|2025-05-03 16:47:...|
|  4|  NULL|    A01Ass|         NULL|        NULL|              NULL|    NULL|           NULL|2025-05-03 16:47:...|
|  1| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:47:...|
|  1| 11001|AW00011001|       Eugene|     Huang  |                 S|   

In [59]:
cust_info_edit_qulity_cst_id_2 = spark.sql('''
    SELECT *
    FROM (
        SELECT ROW_NUMBER() over(partition by cst_id order by cst_create_date DESC) as rn,
        crm.cst_id , crm.cst_key , crm.cst_firstname , crm.cst_lastname , crm.cst_marital_status , crm.cst_gndr,
        crm.cst_create_date , crm.dwh_cst_date
        FROM customer_information as crm 
        ) as t
    WHERE t.cst_id IS NOT NULL AND t.rn = 1
''')

In [60]:
cust_info_edit_qulity_cst_id_2.show()

+---+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| rn|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+---+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|  1| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:47:...|
|  1| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2025-05-03 16:47:...|
|  1| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 16:47:...|
|  1| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
|  1| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
|  1| 11005|AW00011005|        Julio|        Ruiz|                 S|   

In [61]:
cust_info_drop_rn=cust_info_edit_qulity_cst_id_2.drop('rn')

In [62]:
cust_info_drop_rn.show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:47:...|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2025-05-03 16:47:...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 16:47:...|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2025-05-03 16:47:...|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2025-05-03 16:

In [63]:
cust_info_good_qulity_cst_id = cust_info_drop_rn.select('*')\
.groupBy('cst_id')\
.agg(fn.count('*').alias('cst_count'))\
.where((col('cst_count') > 1) & (col('cst_id').isNull())).show()

+------+---------+
|cst_id|cst_count|
+------+---------+
+------+---------+



In [66]:
cust_info_drop_rn.select('*')\
.where(~col('cst_key').like('AW%')).show()

+------+-------+-------------+------------+------------------+--------+---------------+------------+
|cst_id|cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|dwh_cst_date|
+------+-------+-------------+------------+------------------+--------+---------------+------------+
+------+-------+-------------+------------+------------------+--------+---------------+------------+



In [72]:
cust_info_drop_rn.select('cst_firstname')\
.where(col('cst_firstname')!=trim('cst_firstname')).show()

+-------------+
|cst_firstname|
+-------------+
+-------------+



In [70]:
cust_info_remove_spaces = cust_info_drop_rn.withColumn('cst_firstname' , trim(col('cst_firstname')))

In [71]:
cust_info_remove_spaces.show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2025-05-03 16:59:...|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2025-05-03 16:59:...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 16:59:...|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2025-05-03 16:59:...|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2025-05-03 16:59:...|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2025-05-03 16:

In [77]:
cust_info_remove_spaces.select('cst_firstname')\
.where(col('cst_firstname')!=trim('cst_firstname')).show()

+------------+
|cst_lastname|
+------------+
+------------+



In [75]:
cust_info_remove_spaces = cust_info_remove_spaces.withColumn('cst_lastname', trim(col('cst_lastname')))

In [76]:
cust_info_remove_spaces.show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|        Yang|                 M|       M|     2025-10-06|2025-05-03 17:01:...|
| 11001|AW00011001|       Eugene|       Huang|                 S|       M|     2025-10-06|2025-05-03 17:01:...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2025-05-03 17:01:...|
| 11003|AW00011003|      Christy|         Zhu|                 S|       F|     2025-10-06|2025-05-03 17:01:...|
| 11004|AW00011004|    Elizabeth|     Johnson|                 S|       F|     2025-10-06|2025-05-03 17:01:...|
| 11005|AW00011005|        Julio|        Ruiz|                 S|       M|     2025-10-06|2025-05-03 17:

In [78]:
cust_info_remove_spaces_view_stand = cust_info_remove_spaces.createOrReplaceTempView('customer_info')

In [86]:
cust_info_remove_spaces.select('cst_marital_status').distinct().show()

+------------------+
|cst_marital_status|
+------------------+
|                 M|
|                 S|
+------------------+



In [90]:
cust_info_remove_spaces.select('cst_gndr').distinct().show()

+--------+
|cst_gndr|
+--------+
|       F|
|       M|
|    NULL|
+--------+



In [92]:
cust_info_stand_martial_gend = spark.sql('''
    SELECT   crm.cst_id , crm.cst_key , crm.cst_firstname, crm.cst_lastname , 
    CASE
        WHEN UPPER(TRIM(crm.cst_marital_status)) = 'M' THEN 'MARIED'
        WHEN UPPER(TRIM(crm.cst_marital_status)) = 'S' THEN 'SINGLE'  
        ELSE 'N/A'
    END AS cst_marital_status ,

    CASE 
        WHEN UPPER(TRIM(crm.cst_gndr)) = 'M' THEN 'MALE' 
        WHEN UPPER(TRIM(crm.cst_gndr)) = 'F' THEN  'FEMALE' 
        ELSE 'N/A'
    END AS cst_gndr ,
    crm.cst_create_date , crm.dwh_cst_date
    FROM customer_info as crm 
''')

In [93]:
cust_info_stand_martial_gend.show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|        Yang|            MARIED|    MALE|     2025-10-06|2025-05-03 17:11:...|
| 11001|AW00011001|       Eugene|       Huang|            SINGLE|    MALE|     2025-10-06|2025-05-03 17:11:...|
| 11002|AW00011002|        Ruben|      Torres|            MARIED|    MALE|     2025-10-06|2025-05-03 17:11:...|
| 11003|AW00011003|      Christy|         Zhu|            SINGLE|  FEMALE|     2025-10-06|2025-05-03 17:11:...|
| 11004|AW00011004|    Elizabeth|     Johnson|            SINGLE|  FEMALE|     2025-10-06|2025-05-03 17:11:...|
| 11005|AW00011005|        Julio|        Ruiz|            SINGLE|    MALE|     2025-10-06|2025-05-03 17:

In [94]:
cust_info_cahnge_date = cust_info_stand_martial_gend.withColumn('cst_create_date' , to_date(col('cst_create_date')))

In [95]:
cust_info_stand_martial_gend.show()

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|        dwh_cst_date|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+
| 11000|AW00011000|          Jon|        Yang|            MARIED|    MALE|     2025-10-06|2025-05-03 17:12:...|
| 11001|AW00011001|       Eugene|       Huang|            SINGLE|    MALE|     2025-10-06|2025-05-03 17:12:...|
| 11002|AW00011002|        Ruben|      Torres|            MARIED|    MALE|     2025-10-06|2025-05-03 17:12:...|
| 11003|AW00011003|      Christy|         Zhu|            SINGLE|  FEMALE|     2025-10-06|2025-05-03 17:12:...|
| 11004|AW00011004|    Elizabeth|     Johnson|            SINGLE|  FEMALE|     2025-10-06|2025-05-03 17:12:...|
| 11005|AW00011005|        Julio|        Ruiz|            SINGLE|    MALE|     2025-10-06|2025-05-03 17:

### ✅ Now upload the data into mysql rdbms 

In [96]:
cust_info_stand_martial_gend.write.format('jdbc')\
.option('url' , 'jdbc:mysql://mysql_db:3306/mytestdb')\
.option("driver", "com.mysql.cj.jdbc.Driver") \
.option('dbtable' , 'customerinfo') \
.option('user' , 'root')\
.option('password','P@ssw0rd')\
.mode('append') \
.save()